# llminfer OpenAI-Compatible Serving (Colab)

Run `llminfer` as an OpenAI-compatible HTTP API with continuous batching.

## 0) Enable GPU runtime
In Colab: **Runtime -> Change runtime type -> GPU**

In [2]:
!nvidia-smi

Wed Feb 25 15:17:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1) Clone + install with serving extras

In [3]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'  # TODO
TARGET_DIR = '/content/llminfer'

import os, shutil
if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone {REPO_URL} {TARGET_DIR}
%cd /content/llminfer
!pip -q install -U pip
!pip -q install -e '.[serve]'
!pip -q install requests

Cloning into '/content/llminfer'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 82 (delta 34), reused 65 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 381.89 KiB | 21.22 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/llminfer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llminfer (pyproject.toml) ... done


## 2) Start server in a background thread

In [4]:
import threading
import time
import traceback

import requests
import uvicorn
from llminfer import EngineConfig, InferenceEngine
from llminfer.api import create_openai_app
from llminfer.serving import ContinuousBatchScheduler

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'  # smaller/faster for Colab startup

cfg = EngineConfig(
    model_name=MODEL_NAME,
    max_batch_size=16,
    batch_timeout_ms=20,
)
engine = InferenceEngine(cfg)
scheduler = ContinuousBatchScheduler(engine, max_batch_size=16, batch_timeout_ms=20, max_queue_size=1024)
app = create_openai_app(engine=engine, scheduler=scheduler, model_alias='local-llminfer')

server_errors = []

def run_server():
    try:
        uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')
    except Exception:
        server_errors.append(traceback.format_exc())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

base_url = 'http://127.0.0.1:8000'
deadline = time.time() + 300
ready = False

while time.time() < deadline:
    if server_errors:
        raise RuntimeError('Server crashed during startup:\n' + server_errors[-1])
    if not server_thread.is_alive():
        raise RuntimeError('Server thread exited before becoming ready.')
    try:
        r = requests.get(f'{base_url}/healthz', timeout=3)
        if r.ok:
            print('Server ready:', r.json())
            ready = True
            break
    except requests.RequestException:
        pass
    time.sleep(2)

if not ready:
    raise TimeoutError('Server did not become ready in 5 minutes.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Server ready: {'status': 'ok', 'model': 'local-llminfer', 'backend': 'eager', 'queue_size': 0}


## 3) Non-streaming chat completion

In [5]:
import requests

payload = {
  'model': 'local-llminfer',
  'messages': [
      {'role': 'system', 'content': 'You are concise.'},
      {'role': 'user', 'content': 'Tell me about GPUs in 4 bullet points.'}
  ],
  'max_tokens': 160,
  'temperature': 0.2
}
resp = requests.post('http://127.0.0.1:8000/v1/chat/completions', json=payload, timeout=180)
resp.raise_for_status()
resp.json()

{'id': 'chatcmpl-148d03dbc5154e7796998a32',
 'object': 'chat.completion',
 'created': 1772032679,
 'model': 'local-llminfer',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': ' Sure, here are four bullet points on GPUs:\n\n1. **Purpose**: GPUs are specialized hardware designed to perform complex mathematical calculations, such as matrix operations, graphics processing, and deep learning.\n\n2. **Architecture**: GPUs are composed of multiple processing units called GPUs, each with its own dedicated hardware architecture. They are typically composed of a central processing unit (CPU) and multiple GPU cores.\n\n3. **Performance**: GPUs are highly efficient at performing complex calculations, making them ideal for scientific simulations, machine learning, and other high-performance computing tasks.\n\n4. **Scalability**: GPUs can be easily scaled up or down based on the workload, making them suitable for both small and large-scale applications. They can also be 

## 4) Streaming chat completion (SSE)

In [9]:
import json

stream_payload = {
  'model': 'local-llminfer',
  'messages': [{'role': 'user', 'content': 'Explain KV cache in under 80 words.'}],
  'max_tokens': 120,
  'temperature': 0.2,
  'stream': True
}

decoder = json.JSONDecoder()
done = False

with requests.post('http://127.0.0.1:8000/v1/chat/completions', json=stream_payload, stream=True, timeout=180) as r:
    r.raise_for_status()

    for raw_line in r.iter_lines(decode_unicode=True):
        if not raw_line:
            continue

        line = raw_line.strip()
        if not line or 'data:' not in line:
            continue

        # Handle coalesced frames: data: {...}data: {...}
        payloads = [p.strip() for p in line.split('data:') if p.strip()]
        for payload in payloads:
            if payload == '[DONE]':
                done = True
                break

            # Ignore heartbeat/blank escaped payload fragments.
            if payload in {'\\n', '\\n\\n', '"\\n"', '"\\n\\n"'}:
                continue

            s = payload
            while s:
                s = s.lstrip()
                if not s:
                    break
                try:
                    obj, idx = decoder.raw_decode(s)
                except json.JSONDecodeError:
                    # Skip non-JSON fragments silently in notebook output.
                    break

                s = s[idx:]
                delta = obj.get('choices', [{}])[0].get('delta', {}).get('content', '')
                if delta:
                    print(delta, end='', flush=True)

        if done:
            print('\n[done]')
            break


 A cache is a temporary storage device that stores frequently accessed data, such as user data, in memory to reduce disk I/O and improve performance. It uses a hash function to map data to a specific location in memory, allowing for quick access to data. Examples include RAM, SSDs, and disk-based caches. Cache sizes can be adjusted to optimize performance. Cache eviction policies determine how data is removed from the cache when it becomes outdated. Cache hit rates are high when data is frequently accessed, while cache miss rates are low when data is rarely accessed. Cache size and eviction policies can significantly impact performance

## 5) Health and metrics

In [8]:
print(requests.get('http://127.0.0.1:8000/healthz', timeout=20).json())
metrics_text = requests.get('http://127.0.0.1:8000/metrics', timeout=20).text
print('\n'.join(metrics_text.splitlines()[:20]))

{'status': 'ok', 'model': 'local-llminfer', 'backend': 'eager', 'queue_size': 0}
# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 14001.0
python_gc_objects_collected_total{generation="1"} 1523.0
python_gc_objects_collected_total{generation="2"} 626.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 1610.0
python_gc_collections_total{generation="1"} 146.0
python_gc_collections_total{generation="2"} 9.0
# HELP python_info Python platform information
